# Python Analysis of Electron Microscopy Data
---
*Introduction to Image Analysis Workshop*

*Tom Slater (slatert2@cardiff.ac.uk)*

*Intro to building image analysis pipelines for electron microscopy data with Python*

*CC-BY-SA-4.0 license: creativecommons.org/licenses/by-sa/4.0/*

---

# Introduction

In this notebook, we will explore two important concepts in image analysis:

- Watershedding segmentation using scikit-image
- Fourier transforms of images using NumPy

Watershedding is commonly used to separate touching or overlapping objects in microscopy and materials imaging. Fourier transforms are used to analyse periodicity, spatial frequency, and image structure.

By the end of this notebook, you should be able to:

- Segment touching objects using a watershed algorithm
- Generate markers for watershed segmentation
- Compute and visualise a 2D Fourier transform
- Interpret frequency-domain information in images

## Importing libraries

We first import the libraries required for the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy import ndimage as ndi

from skimage import data
from skimage import filters
from skimage import feature
from skimage import segmentation
from skimage import measure
from skimage import morphology
from skimage.color import label2rgb
from skimage import io

import pandas as pd

## Part 1 — Watershed segmentation for EM data
We'll first load a high-angle annular dark field (HAADF) scanning transmission electron microscope (STEM) image of SiO2 nanoparticles. The vast majority of electron microscopy images are single channel data, i.e. they are already grayscale images with only two dimensions (y, x).

In [ ]:
im = io.imread("../../SiO2 HAADF Image.tiff")

print('Image dimensions:', im.shape)
print('Image type:', type(im))

In [ ]:
# display image
fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(im, cmap='gray')
ax.set_title('SiO2 nanoparticles')
ax.axis('off')
plt.tight_layout()

## Thresholding the image

We first convert the image into a binary mask using Otsu thresholding (as seen in the previous session).

In [ ]:
threshold = filters.threshold_otsu(im)

im_thresh = im > threshold

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(im_thresh, cmap='gray')
ax.set_title('Binary image')
ax.axis('off')

If we apply the same labelling as completed previously, we are not able to accurately label each nanoparticle due to the connectedness of each label. We could try exploring different thresholding algorithms, but it will be difficult to separate touching nanoparticles.

In [ ]:
# label objects and visualise the result
labels = measure.label(im_thresh)

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(labels, cmap='jet')
ax.axis('off')

In [ ]:
# count the objects - find the maximum integer assigned to a label!
print('There are', labels.max(), 'objects in the image')

## Picking out seeds for the watershed algorithm

The watershed algorithm often uses a distance transform to locate the centres of objects. The local maxima of the distance transform can then be used to seed the watershed algorithm.

In [ ]:
distance = ndi.distance_transform_edt(im_thresh)

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(distance, cmap='magma')
ax.set_title('Distance transform')
ax.axis('off')

We now identify peaks in the distance transform.

These peaks will act as markers for the watershed segmentation.

In [ ]:
coords = feature.peak_local_max(
    distance,
    min_distance=3,
    labels=binary
)

mask = np.zeros(distance.shape, dtype=bool)
mask[tuple(coords.T)] = True

markers, _ = ndi.label(mask)

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(mask, cmap='gray')
ax.set_title('Marker positions')
ax.axis('off')

## Applying watershed segmentation

We can now apply the watershed algorithm. The labels object that is returned by the watershed function is equivalent to that returned by <code>measure.label</code>, although in this case we now have many more labels!

In [ ]:
labels_watershed = segmentation.watershed(
    -distance,
    markers,
    mask=binary
)

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(labels_watershed, cmap='nipy_spectral')
ax.set_title('Watershed segmentation')
ax.axis('off')

In [ ]:
# count the objects - find the maximum integer assigned to a label!
print('There are', labels_watershed.max(), 'objects in the image')

## Measuring segmented objects

We can extract measurements from the labelled regions in exactly the same manner as previously.

In [ ]:
# measure properties
props = measure.regionprops_table(labels, im, properties=['area', 'centroid', 'eccentricity'])
props_df = pd.DataFrame(props)

props_df.head(5)

In [ ]:
# display some results
fig, axs = plt.subplots(1, 2, figsize=(4,3))

axs[0].boxplot(props_df['area'])
axs[0].set_title('Particle area (px)')

axs[1].boxplot(props_df['eccentricity'])
axs[1].set_title('Particle eccentricity')

plt.tight_layout()

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
The effectiveness of the watershedding algorithm is highly dependent on the choice of minimum distance between local maxima. Try changing the minimum distance in the code below to 2 or 4 pixels and plot the labels to see what happens to the connectivity of labels. </div>

In [ ]:
coords = feature.peak_local_max(
    distance,
    min_distance=,#Enter a new minimum distance here
    labels=binary
)

mask = np.zeros(distance.shape, dtype=bool)
mask[tuple(coords.T)] = True

markers, _ = ndi.label(mask)

In [ ]:
labels_watershed = segmentation.watershed(
    -distance,
    markers,
    mask=binary
)

#Plot the data below

# Part 2 — Fourier transforms of images
## Introduction

Fourier transforms convert an image from:

Real space (position)

into:

Frequency space (spatial frequencies)

This allows us to detect:

- Periodic structures
- Repeating patterns
- Directional information
- Noise frequencies

## Loading data

Let's load a HAADF-STEM image of gold nanoparticles at atomic-resolution.

In [ ]:
im = io.imread("../../au_carbon.tif")

print('Image dimensions:', im.shape)
print('Image type:', type(im))

In [ ]:
# display image
fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(im, cmap='gray')
ax.set_title('Au nanoparticles')
ax.axis('off')
plt.tight_layout()

Again, we note that the image is natively grayscale (i.e. single channel). Here, the image has also been saved with an annotation overlayed (the scalebar), which is particularly common when dealing with SEM data.

## Computing the Fourier transform

At this stage, let's compute the Fourier transform for the whole image using NumPy's <code>fft.fft2</code> function. This function is specifically for two-dimensional images.

In the code below, we then apply a shift to ensure that low frequency information is in the centre of our Fourier transform (as is standard). 

The output of the Fourier transform contains both the magnitude and phase in a complex array. We use the <code>abs</code> function to obtain solely the magnitude component, and then take the natural logarithm of this array.

In [ ]:
fft = np.fft.fft2(im)
fft_shifted = np.fft.fftshift(fft)

magnitude = np.abs(fft_shifted)
log_magnitude = np.log1p(magnitude)

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(log_magnitude, cmap='inferno')
ax.set_title('Fourier transform magnitude')
ax.axis('off')

The Fourier transform of the entire image contains a number of peaks corresponding to the freqeuncy of lattice information we can see in the image. These come from many different nanoparticles and it would be difficult for us to work out which peak came from which particle.

We also have undesirable vertical and horizontal lines, which come from the sharp borders of our image and the overlayed scale bar.

All in all it's a bit of a mess! Instead, we could crop our image using array slicing to take the Fourier transform of just a single nanoparticle.

## Fourier Transform of Single Nanoparticle

Let's use NumPy's inbuilt array slicing to crop around a specific nanoparticle. Slicing works using square brackets at the end of an array. For example, to take the top left quarter of our image we would specify:

In [ ]:
cropped_im = im[0:303,0:305]
#The two values either side of the colon give the start and end positions for slicing.

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(cropped_im)
ax.set_title('Cropped image')
ax.axis('off')

Now let's crop around one nanoparticle only, to obtain the Fourier transform from a single nanoparticle.

In [ ]:
particle_im = im[180:360,10:190]

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(particle_im)
ax.set_title('Cropped image')
ax.axis('off')

In [ ]:
fft = np.fft.fft2(particle_im)
fft_shifted = np.fft.fftshift(fft)

magnitude = np.abs(fft_shifted)
log_magnitude = np.log1p(magnitude)

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(log_magnitude, cmap='inferno')
ax.set_title('Fourier transform magnitude')
ax.axis('off')

We now have a Fourier transform that corresponds to the frequency information from a single nanoparticle. If we pick out the peaks in the FFT, we can work out what frequencies these correspond to and therefore determine the spacings of lattice planes / atomic columns in our image.

To do this, we first use the same <code>peak_local_max</code> function we used when watershedding to find the peaks in the FFT. We'll use array slicing to just select the 3 most intense peaks to plot.

In [ ]:
coordinates = feature.peak_local_max(magnitude, min_distance=20)

In [ ]:
plt.figure()
plt.imshow(log_magnitude)
plt.scatter(coordinates[:3,1], coordinates[:3,0], s = 50, facecolors = 'none', edgecolors = 'r');

Now we have the peak locations, we can extract the distance between two symmetric peaks to calculate the distance between them in pixels, which we can then convert back to real space distance if we know the pixel size in our original image (which here is approximately 0.032 nm).

In [ ]:
pix = np.sqrt((coordinates[1,1]-coordinates[2,1])**2+(coordinates[1,0]-coordinates[2,0])**2)/2
spacing = 180*0.032/pix

print("The spacing found from the Fourier transform is:", spacing, "nm")

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
Above, we took the Fourier transform of one of the nanoparticles and calculated the most prominent lattice spacing. Try to crop another of the nanoparticles in the image and calculate the most prominent lattice spacing using its Fourier transform.</div>

In [ ]:
particle_im_test = im[?:?,?:?] #Replace the question marks with values to test

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(particle_im_test)
ax.set_title('Cropped image')
ax.axis('off')

In [ ]:
#Perform a Fourier transform and use this to calculate lattice spacing
